# Unit 2.4: Morphological Operations

This notebook covers morphological image processing:
- Erosion and Dilation
- Opening and Closing
- Morphological Gradient
- Top-Hat and Black-Hat transforms
- Practical applications

## Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Image

In [ ]:
# Load image and convert to binary
img_bgr = cv2.imread('sample.jpeg')
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

# Create binary image
binary_img = cv2.threshold(img_gray, 128, 255, cv2.THRESH_BINARY)[1]

print(f"Binary Image Shape: {binary_img.shape}")
print(f"White (255) pixels: {np.count_nonzero(binary_img)} ({100*np.count_nonzero(binary_img)/binary_img.size:.2f}%)")
print(f"Black (0) pixels: {np.count_nonzero(255-binary_img)} ({100*np.count_nonzero(255-binary_img)/binary_img.size:.2f}%)")

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(img_gray, cmap='gray')
plt.title('Original Grayscale')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(binary_img, cmap='gray')
plt.title('Binary Image (Threshold=128)')
plt.axis('off')

plt.tight_layout()
plt.show()

## 2. Structuring Elements

In [ ]:
# Create different structuring elements
rectangular_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
ellipse_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
cross_kernel = cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))

# Manual kernels
custom_kernel = np.array([[0, 1, 0],
                          [1, 1, 1],
                          [0, 1, 0]], dtype=np.uint8)

plt.figure(figsize=(12, 3))

plt.subplot(1, 4, 1)
plt.imshow(rectangular_kernel, cmap='gray')
plt.title('Rectangular Kernel (5x5)')
for i in range(rectangular_kernel.shape[0]):
    for j in range(rectangular_kernel.shape[1]):
        val = rectangular_kernel[i, j]
        plt.text(j, i, str(val), ha='center', va='center', color='white' if val else 'black')

plt.subplot(1, 4, 2)
plt.imshow(ellipse_kernel, cmap='gray')
plt.title('Ellipse Kernel (5x5)')
for i in range(ellipse_kernel.shape[0]):
    for j in range(ellipse_kernel.shape[1]):
        val = ellipse_kernel[i, j]
        plt.text(j, i, str(val), ha='center', va='center', color='white' if val else 'black')

plt.subplot(1, 4, 3)
plt.imshow(cross_kernel, cmap='gray')
plt.title('Cross Kernel (5x5)')
for i in range(cross_kernel.shape[0]):
    for j in range(cross_kernel.shape[1]):
        val = cross_kernel[i, j]
        plt.text(j, i, str(val), ha='center', va='center', color='white' if val else 'black')

plt.subplot(1, 4, 4)
plt.imshow(custom_kernel, cmap='gray')
plt.title('Custom Kernel')
for i in range(custom_kernel.shape[0]):
    for j in range(custom_kernel.shape[1]):
        val = custom_kernel[i, j]
        plt.text(j, i, str(val), ha='center', va='center', color='white' if val else 'black')

plt.tight_layout()
plt.show()

print("Structuring Element Shapes:")
print("- Rectangular: Includes all neighbors in rectangle")
print("- Ellipse/Circle: Includes circular neighborhood")
print("- Cross: Only 4-connected neighbors (horizontal + vertical)")
print("- Custom: User-defined patterns")

## 3. Erosion

In [ ]:
# Create test image with noise
noise_mask = np.random.rand(*binary_img.shape) < 0.05
noisy_binary = binary_img.copy()
noisy_binary[noise_mask] = 255 - noisy_binary[noise_mask]

# Apply erosion with different iterations
erosion_results = []
iterations_list = [1, 2, 3, 5]

kernels = [
    ('Rect 3x3', cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))),
    ('Ellipse 5x5', cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
]

fig, axes = plt.subplots(len(kernels), len(iterations_list) + 1, figsize=(15, 6))

for k_idx, (kernel_name, kernel) in enumerate(kernels):
    # Show original
    axes[k_idx, 0].imshow(noisy_binary, cmap='gray')
    axes[k_idx, 0].set_title(f'Original\n{kernel_name}')
    axes[k_idx, 0].axis('off')
    
    # Apply erosion
    for it_idx, iterations in enumerate(iterations_list):
        eroded = cv2.erode(noisy_binary, kernel, iterations=iterations)
        erosion_results.append(eroded)
        
        axes[k_idx, it_idx + 1].imshow(eroded, cmap='gray')
        axes[k_idx, it_idx + 1].set_title(f'Erosion iter={iterations}')
        axes[k_idx, it_idx + 1].axis('off')
        
        white_pixels = np.count_nonzero(eroded)
        axes[k_idx, it_idx + 1].text(0.5, -0.1, f'White: {white_pixels}', 
                                     transform=axes[k_idx, it_idx + 1].transAxes,
                                     ha='center', fontsize=8)

plt.tight_layout()
plt.suptitle('Erosion Operation: Shrinks white regions', fontsize=14, y=1.00)
plt.show()

print("\nErosion Properties:")
print("- Removes small white objects (noise)")
print("- Shrinks white regions")
print("- Output: min(kernel neighbors)")
print(f"- Original white pixels: {np.count_nonzero(noisy_binary)}")
for it, iter_val in enumerate(iterations_list):
    print(f"- After {iter_val} erosion(s): {np.count_nonzero(erosion_results[it])} pixels")

## 4. Dilation

In [ ]:
# Apply dilation with different iterations
dilation_results = []

fig, axes = plt.subplots(len(kernels), len(iterations_list) + 1, figsize=(15, 6))

for k_idx, (kernel_name, kernel) in enumerate(kernels):
    # Show original
    axes[k_idx, 0].imshow(noisy_binary, cmap='gray')
    axes[k_idx, 0].set_title(f'Original\n{kernel_name}')
    axes[k_idx, 0].axis('off')
    
    # Apply dilation
    for it_idx, iterations in enumerate(iterations_list):
        dilated = cv2.dilate(noisy_binary, kernel, iterations=iterations)
        dilation_results.append(dilated)
        
        axes[k_idx, it_idx + 1].imshow(dilated, cmap='gray')
        axes[k_idx, it_idx + 1].set_title(f'Dilation iter={iterations}')
        axes[k_idx, it_idx + 1].axis('off')
        
        white_pixels = np.count_nonzero(dilated)
        axes[k_idx, it_idx + 1].text(0.5, -0.1, f'White: {white_pixels}', 
                                     transform=axes[k_idx, it_idx + 1].transAxes,
                                     ha='center', fontsize=8)

plt.tight_layout()
plt.suptitle('Dilation Operation: Expands white regions', fontsize=14, y=1.00)
plt.show()

print("\nDilation Properties:")
print("- Fills small black holes")
print("- Expands white regions")
print("- Output: max(kernel neighbors)")
print(f"- Original white pixels: {np.count_nonzero(noisy_binary)}")
for it in range(len(iterations_list)):
    print(f"- After dilation(s): {np.count_nonzero(dilation_results[it + len(iterations_list)*0])} pixels")

## 5. Opening (Erosion → Dilation)

In [ ]:
# Apply opening operation
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

opened = cv2.morphologyEx(noisy_binary, cv2.MORPH_OPEN, kernel)
# Manual opening for comparison
manual_opened = cv2.dilate(cv2.erode(noisy_binary, kernel), kernel)

# Step-by-step visualization
eroded = cv2.erode(noisy_binary, kernel)

plt.figure(figsize=(15, 3))

plt.subplot(1, 5, 1)
plt.imshow(noisy_binary, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 5, 2)
plt.imshow(eroded, cmap='gray')
plt.title('After Erosion')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.imshow(opened, cmap='gray')
plt.title('Opening Result\n(Erosion→Dilation)')
plt.axis('off')

# Show difference
opening_effect = cv2.absdiff(noisy_binary, opened)
plt.subplot(1, 5, 4)
plt.imshow(opening_effect, cmap='hot')
plt.title('Removed Objects')
plt.colorbar()

# Show reconstruction
reconstruction = cv2.absdiff(opened, eroded)
plt.subplot(1, 5, 5)
plt.imshow(reconstruction, cmap='hot')
plt.title('Reconstruction Effect')
plt.colorbar()

plt.tight_layout()
plt.show()

print("\nOpening (Erosion → Dilation):")
print(f"- Original white pixels: {np.count_nonzero(noisy_binary)}")
print(f"- After opening: {np.count_nonzero(opened)}")
print(f"- Removed: {np.count_nonzero(noisy_binary) - np.count_nonzero(opened)} pixels")
print("- Use: Remove small noise objects while preserving large structures")

## 6. Closing (Dilation → Erosion)

In [ ]:
# Create image with holes
img_with_holes = binary_img.copy()

# Add some black holes in white regions
for _ in range(20):
    x, y = np.random.randint(50, img_with_holes.shape[1]-50), np.random.randint(50, img_with_holes.shape[0]-50)
    cv2.circle(img_with_holes, (x, y), 5, 0, -1)

# Apply closing operation
closed = cv2.morphologyEx(img_with_holes, cv2.MORPH_CLOSE, kernel)

# Step-by-step visualization
dilated = cv2.dilate(img_with_holes, kernel)

plt.figure(figsize=(15, 3))

plt.subplot(1, 5, 1)
plt.imshow(img_with_holes, cmap='gray')
plt.title('Original (With Holes)')
plt.axis('off')

plt.subplot(1, 5, 2)
plt.imshow(dilated, cmap='gray')
plt.title('After Dilation')
plt.axis('off')

plt.subplot(1, 5, 3)
plt.imshow(closed, cmap='gray')
plt.title('Closing Result\n(Dilation→Erosion)')
plt.axis('off')

# Show difference
closing_effect = cv2.absdiff(img_with_holes, closed)
plt.subplot(1, 5, 4)
plt.imshow(closing_effect, cmap='hot')
plt.title('Filled Holes')
plt.colorbar()

# Show reconstruction
reconstruction = cv2.absdiff(closed, dilated)
plt.subplot(1, 5, 5)
plt.imshow(reconstruction, cmap='hot')
plt.title('Reconstruction Effect')
plt.colorbar()

plt.tight_layout()
plt.show()

print("\nClosing (Dilation → Erosion):")
print(f"- Original white pixels: {np.count_nonzero(img_with_holes)}")
print(f"- After closing: {np.count_nonzero(closed)}")
print(f"- Filled: {np.count_nonzero(closed) - np.count_nonzero(img_with_holes)} pixels")
print("- Use: Fill small holes while preserving object size")

## 7. Morphological Gradient, Top-Hat & Black-Hat

In [ ]:
# Morphological operations
gradient = cv2.morphologyEx(binary_img, cv2.MORPH_GRADIENT, kernel)
tophat = cv2.morphologyEx(binary_img, cv2.MORPH_TOPHAT, kernel)
blackhat = cv2.morphologyEx(binary_img, cv2.MORPH_BLACKHAT, kernel)

# Manual calculations for reference
dilated = cv2.dilate(binary_img, kernel)
eroded = cv2.erode(binary_img, kernel)
manual_gradient = cv2.absdiff(dilated, eroded)

opened = cv2.morphologyEx(binary_img, cv2.MORPH_OPEN, kernel)
manual_tophat = cv2.absdiff(binary_img, opened)

closed = cv2.morphologyEx(binary_img, cv2.MORPH_CLOSE, kernel)
manual_blackhat = cv2.absdiff(closed, binary_img)

plt.figure(figsize=(15, 4))

plt.subplot(1, 4, 1)
plt.imshow(binary_img, cmap='gray')
plt.title('Original')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(gradient, cmap='gray')
plt.title('Morphological Gradient\n(Dilation - Erosion)')
plt.axis('off')
plt.text(0.5, -0.1, f'Use: Extract edges and boundaries', 
         transform=plt.gca().transAxes, ha='center', fontsize=9)

plt.subplot(1, 4, 3)
plt.imshow(tophat, cmap='gray')
plt.title('Top-Hat\n(Original - Opening)')
plt.axis('off')
plt.text(0.5, -0.1, f'Use: Extract small objects', 
         transform=plt.gca().transAxes, ha='center', fontsize=9)

plt.subplot(1, 4, 4)
plt.imshow(blackhat, cmap='gray')
plt.title('Black-Hat\n(Closing - Original)')
plt.axis('off')
plt.text(0.5, -0.1, f'Use: Extract small holes', 
         transform=plt.gca().transAxes, ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print("Advanced Morphological Operations:")
print("\n1. Morphological Gradient (Dilation - Erosion):")
print(f"   - Non-zero pixels: {np.count_nonzero(gradient)}")
print("   - Use: Outline/edge extraction")

print("\n2. Top-Hat (Original - Opening):")
print(f"   - Non-zero pixels: {np.count_nonzero(tophat)}")
print("   - Use: Extract small bright objects")

print("\n3. Black-Hat (Closing - Original):")
print(f"   - Non-zero pixels: {np.count_nonzero(blackhat)}")
print("   - Use: Extract small dark objects/holes")

## 8. Practical Applications

In [ ]:
# Application 1: Noise removal from binary image
print("Application 1: Noise Removal")
print("-" * 40)

noisy = noisy_binary.copy()
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
denoised = cv2.morphologyEx(noisy, cv2.MORPH_OPEN, kernel)  # Remove noise
denoised = cv2.morphologyEx(denoised, cv2.MORPH_CLOSE, kernel)  # Fill holes

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(noisy, cmap='gray')
plt.title(f'Noisy ({np.count_nonzero(noisy)} white pixels)')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(cv2.morphologyEx(noisy, cv2.MORPH_OPEN, kernel), cmap='gray')
plt.title('After Opening')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(denoised, cmap='gray')
plt.title(f'Denoised ({np.count_nonzero(denoised)} white pixels)')
plt.axis('off')

plt.tight_layout()
plt.show()

print(f"Original noisy pixels: {np.count_nonzero(noisy)}")
print(f"After denoising: {np.count_nonzero(denoised)}")
print(f"Improvement: {100*(np.count_nonzero(noisy)-np.count_nonzero(denoised))/np.count_nonzero(noisy):.2f}%")

## Application 2: Connected Component Analysis

In [ ]:
# Find connected components
num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(denoised, connectivity=8)

print(f"\nApplication 2: Connected Component Analysis")
print("-" * 40)
print(f"Number of components: {num_labels}")
print(f"\nComponent Statistics:")
print(f"{'ID':<5} {'Area':<10} {'Centroid':<20}")
print("-" * 35)

for i in range(1, num_labels):  # Skip background (0)
    area = stats[i, cv2.CC_STAT_AREA]
    centroid = centroids[i]
    print(f"{i:<5} {area:<10} ({centroid[0]:.1f}, {centroid[1]:.1f})")

# Visualize components with different colors
color_labels = np.zeros((*denoised.shape, 3), dtype=np.uint8)
for i in range(1, num_labels):
    color = (np.random.randint(0, 255), np.random.randint(0, 255), np.random.randint(0, 255))
    color_labels[labels == i] = color

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.imshow(denoised, cmap='gray')
plt.title('Denoised Binary Image')
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(color_labels)
plt.title(f'Connected Components ({num_labels-1} objects)')
plt.axis('off')

plt.tight_layout()
plt.show()

## Summary: Morphological Operations Reference

### Basic Operations:

| Operation | Formula | Effect |
|-----------|---------|--------|
| Erosion | min(kernel) | Shrinks white, expands black |
| Dilation | max(kernel) | Expands white, shrinks black |
| Opening | Erosion → Dilation | Remove small noise |
| Closing | Dilation → Erosion | Fill small holes |
| Gradient | Dilation - Erosion | Extract boundaries |
| Top-Hat | Original - Opening | Extract small objects |
| Black-Hat | Closing - Original | Extract small holes |

### Key Principles:
1. **Erosion** removes small white objects
2. **Dilation** fills small black holes
3. **Opening** = Erosion→Dilation (preserve large objects)
4. **Closing** = Dilation→Erosion (preserve structure)
5. **Kernel shape** determines behavior

### Applications:
- Image denoising
- Object detection
- Feature extraction
- Image segmentation
- Binary image analysis